# CMS Dataset Variable Exploration

This notebook queries a small sample from the public CMS dataset to inspect its structure before formal analysis. Run the cells from top to bottom to retrieve the sample, list the variables, and inspect the first row. The sample remains in memory and is not saved locally.

In [ ]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

URL = "https://data.cms.gov/data-api/v1/dataset/690ddc6c-2767-4618-b277-420ffb2bf27c/data"
SAMPLE_SIZE = 10  # Change this value to inspect a different sample size.

query_url = f"{URL}?{urlencode({'size': SAMPLE_SIZE})}"
with urlopen(query_url, timeout=30) as response:
    records = json.load(response)

if not records:
    raise ValueError("The API returned no records.")

print(f"Loaded {len(records)} rows.")

## Variables Returned by the CMS API

In [ ]:
columns = list(records[0].keys())

print(f"Total columns: {len(columns)}")
for number, column in enumerate(columns, start=1):
    print(f"{number}. {column}")

## First Sample Row

In [ ]:
first_row = records[0]

for column, value in first_row.items():
    print(f"{column}: {value}")

# Dimensions and Measures

A **dimension** identifies, describes, or categorizes an observation. A **measure** represents a quantitative value that can be meaningfully aggregated, compared, or summarized. A variable can contain numbers and still be a dimension if those numbers function as identifiers or categories.

The roles below use the official CMS data dictionary and are limited to variables observed in the API sample.

| Variable | Role | Reason |
| -------- | ---- | ------ |
| `Rndrng_Prvdr_CCN` | Identifier | CMS Certification Number identifies the rendering hospital provider; it should remain text to preserve leading zeros. |
| `Rndrng_Prvdr_Org_Name` | Descriptive attribute | Human-readable organization name associated with the provider CCN. |
| `Rndrng_Prvdr_City` | Dimension | Categorizes the provider by its reported city. |
| `Rndrng_Prvdr_St` | Descriptive attribute | Reported provider street address; useful for description, not aggregation. |
| `Rndrng_Prvdr_State_FIPS` | Dimension | Standard state geographic code; numeric characters represent a category, not a quantity. |
| `Rndrng_Prvdr_Zip5` | Dimension | Five-digit postal geography used for grouping or filtering, not arithmetic. |
| `Rndrng_Prvdr_State_Abrvtn` | Dimension | Readable state category for the provider location. |
| `Rndrng_Prvdr_RUCA` | Dimension | Rural-Urban Commuting Area code is a categorical geographic classification. |
| `Rndrng_Prvdr_RUCA_Desc` | Descriptive attribute | Human-readable label for the RUCA code. |
| `DRG_Cd` | Identifier | Identifies the MS-DRG service category within the reporting year. |
| `DRG_Desc` | Descriptive attribute | Human-readable clinical and severity label associated with the DRG code. |
| `Tot_Dschrgs` | Measure | Count of reported inpatient discharges for the provider and DRG. |
| `Avg_Submtd_Cvrd_Chrg` | Measure | Average submitted covered charge for the provider and DRG. |
| `Avg_Tot_Pymt_Amt` | Measure | Average total payment amount for the provider and DRG. |
| `Avg_Mdcr_Pymt_Amt` | Measure | Average Medicare payment amount for the provider and DRG. |

# Primary Key Hypothesis

A primary key uniquely identifies each row in a table. The apparent dataset grain is **one hospital + one DRG**, so the initial candidate key combines the provider identifier and the service-category identifier.

> **Hypothesis:** the combination of provider CCN and DRG code uniquely identifies each record in the current 2024 provider-and-service dataset.

This is a hypothesis, not a confirmed fact. The test below covers only the API sample currently loaded in memory.

## Test: Composite-Key Uniqueness in the Current Sample

The test compares the sample row count with the number of distinct `Rndrng_Prvdr_CCN` + `DRG_Cd` combinations and inspects any duplicated combinations.

In [ ]:
import pandas as pd

sample_df = pd.DataFrame(records)
key_columns = ["Rndrng_Prvdr_CCN", "DRG_Cd"]

total_row_count = len(sample_df)
unique_key_count = sample_df[key_columns].drop_duplicates().shape[0]
duplicate_mask = sample_df.duplicated(subset=key_columns, keep=False)
duplicate_rows = sample_df.loc[duplicate_mask].sort_values(key_columns)
duplicate_combination_count = duplicate_rows[key_columns].drop_duplicates().shape[0]

print("Validation scope: current API sample only")
print(f"Total rows: {total_row_count}")
print(f"Unique provider + DRG combinations: {unique_key_count}")
print(f"Duplicate combinations: {duplicate_combination_count}")

if duplicate_rows.empty:
    print("Result: no duplicate key combinations were found in this sample.")
else:
    print("Result: duplicate key combinations were found in this sample.")
    display(duplicate_rows)

## Result and Interpretation

**Sample result:** the current 10-row API sample contains 10 unique provider-DRG combinations and no duplicate combinations.

**Interpretation:** the result is consistent with the candidate key, but it does not validate uniqueness across the complete 2024 dataset. The hypothesis remains open until the full dataset is acquired and tested.

## Current Data Grain Hypothesis

> Based on the CMS documentation and the current API sample, one row appears to represent one hospital-provider and one DRG combination for the selected reporting year. This grain will be formally validated once the complete dataset is acquired.

# Full Dataset Retrieval

The CMS Data API returns paginated responses rather than the entire dataset in one response. The `size` parameter specifies how many rows to request, while `offset` specifies the zero-based starting position for that request. CMS allows a maximum page size of 5,000 rows.

**Reference (APA 7):** Centers for Medicare & Medicaid Services. (n.d.). *API documentation*. Data.CMS.gov. Retrieved September 5, 2026, from https://data.cms.gov/api-docs

## Step 1: Determine the Expected Row Count

The dataset statistics endpoint supplies the current record count. Reading it first avoids hard-coding a row total that may become outdated.

In [ ]:
import requests

STATS_URL = f"{URL}/stats"

stats_response = requests.get(STATS_URL, timeout=30)
stats_response.raise_for_status()
stats = stats_response.json()

print("CMS stats response:")
display(stats)

expected_record_count = int(stats["total_rows"])
print(f"Expected total records: {expected_record_count:,}")

## Step 2: Understand the Pagination Pattern

```text
size = 5000, offset = 0
size = 5000, offset = 5000
size = 5000, offset = 10000
...
```

Each request starts where the previous batch ended. Pagination is required because the expected dataset is larger than the API's 5,000-row maximum page size.

## Step 3: Retrieve All Records

This loop requests one batch at a time, checks every HTTP response, advances the offset explicitly, and keeps the raw JSON records in one list. No cleaning or type conversion occurs during retrieval.

In [ ]:
BATCH_SIZE = 5_000
all_records = []
offset = 0

while len(all_records) < expected_record_count:
    remaining_records = expected_record_count - len(all_records)
    requested_size = min(BATCH_SIZE, remaining_records)
    params = {"size": requested_size, "offset": offset}

    page_response = requests.get(URL, params=params, timeout=60)
    page_response.raise_for_status()
    batch = page_response.json()

    if not isinstance(batch, list):
        raise TypeError("Expected each API page to be a JSON list.")
    if not batch:
        raise RuntimeError("The API returned an empty page before the expected total was reached.")

    all_records.extend(batch)
    offset += len(batch)

    print(
        f"Retrieved {len(all_records):,} of {expected_record_count:,} records "
        f"(latest batch: {len(batch):,})."
    )

print("Full retrieval loop finished.")

## Step 4: Convert the Records to pandas

The raw records are combined before DataFrame creation. Only the shape, a small preview, and the column names are displayed.

In [ ]:
full_df = pd.DataFrame(all_records)

print(f"DataFrame shape: {full_df.shape}")
display(full_df.head())
print("Column names:")
print(full_df.columns.tolist())

## Retrieval Validation

**Test:** compare the CMS expected record count with the number of rows in the DataFrame. The cell raises an error if they differ, which prevents a normal top-to-bottom run from continuing to key validation with incomplete data.

In [ ]:
retrieved_record_count = len(full_df)
retrieval_complete = retrieved_record_count == expected_record_count

print(f"CMS expected records: {expected_record_count:,}")
print(f"DataFrame rows: {retrieved_record_count:,}")
print(f"Counts match: {retrieval_complete}")

if not retrieval_complete:
    raise RuntimeError(
        "Retrieval is incomplete. Do not continue to primary-key validation."
    )

print("Result: all records expected by the CMS stats endpoint were retrieved.")

# Primary Key Validation

**Hypothesis:** `Rndrng_Prvdr_CCN` + `DRG_Cd` uniquely identifies each row in the complete retrieved 2024 dataset.

**Test:** count all rows, distinct provider-DRG combinations, and combinations that occur more than once. The interpretation is printed only after the complete-data test runs.

In [ ]:
if "retrieval_complete" not in globals() or not retrieval_complete:
    raise RuntimeError(
        "Complete and validate the full retrieval before testing the candidate key."
    )

candidate_key = ["Rndrng_Prvdr_CCN", "DRG_Cd"]
total_rows = len(full_df)

provider_drg_counts = (
    full_df.groupby(candidate_key, dropna=False)
    .size()
    .reset_index(name="row_count")
)

unique_provider_drg_count = len(provider_drg_counts)
duplicated_provider_drg = provider_drg_counts.loc[
    provider_drg_counts["row_count"] > 1
]
duplicated_combination_count = len(duplicated_provider_drg)

print(f"Total rows: {total_rows:,}")
print(f"Unique provider-DRG combinations: {unique_provider_drg_count:,}")
print(f"Duplicated provider-DRG combinations: {duplicated_combination_count:,}")

if duplicated_combination_count > 0:
    print("Result: duplicate provider-DRG combinations exist.")
    print("Interpretation: the current grain hypothesis must be reconsidered.")
    display(duplicated_provider_drg.head(10))
else:
    print("Result: the candidate combination is unique in the complete retrieved 2024 dataset.")
    print(
        "Interpretation: provider CCN + DRG code is a validated natural candidate key "
        "for this dataset. No database primary-key design decision is made here."
    )